In [104]:
import pandas as pd
import nltk
from nltk.tokenize import WordPunctTokenizer
from nltk.util import pad_sequence
from nltk.util import bigrams
from nltk.util import ngrams
from nltk.util import everygrams
from nltk.lm.preprocessing import pad_both_ends
from nltk.lm.preprocessing import flatten
from nltk.probability import FreqDist

df_corpus = pd.read_csv("./ressources/train.csv")

def remove_html_tags(text):
  import re
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)

df_corpus['Text'] = df_corpus['Text'].apply(lambda x: remove_html_tags(x))
df_corpus['Tokens'] = df_corpus['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))


In [105]:
tokens = [token for sublist in df_corpus['Tokens'] for token in sublist]
fdist = FreqDist(tokens)

vocab = {token: freq for token, freq in fdist.items() if freq >= 5}
sorted_vocab = sorted(vocab.items(), key=lambda x: (-x[1], x[0]))
top_10_tokens = sorted_vocab[:10]
print(top_10_tokens)

[('.', 130553), (',', 51942), (';', 46143), (':', 44383), ('(', 43460), ('the', 39084), ('to', 35027), ('I', 34670), ('-', 30508), ('=', 29202)]


In [106]:
total_tokens = len(tokens)

oov_tokens = [token for token in tokens if token not in vocab]

num_oov_tokens = len(oov_tokens)

oov_rate = (num_oov_tokens / total_tokens) * 100

print(f"OOV rate: {oov_rate:.3f}%")


OOV rate: 4.826%


In [107]:
from nltk.util import bigrams
tokens_question = df_corpus['Tokens'].iloc[2673]
print("tokens_question", tokens_question)
padded_question = ['<s>'] + tokens_question + ['</s>']
print("padded_question", padded_question)
padded_bigrams = list(bigrams(padded_question))
print(padded_bigrams)




tokens_question ['create', 'table', 'empty', 'How', 'can', 'you', 'create', 'an', 'empty', 'table', 'from', 'an', 'existing', 'table', '?']
padded_question ['<s>', 'create', 'table', 'empty', 'How', 'can', 'you', 'create', 'an', 'empty', 'table', 'from', 'an', 'existing', 'table', '?', '</s>']
[('<s>', 'create'), ('create', 'table'), ('table', 'empty'), ('empty', 'How'), ('How', 'can'), ('can', 'you'), ('you', 'create'), ('create', 'an'), ('an', 'empty'), ('empty', 'table'), ('table', 'from'), ('from', 'an'), ('an', 'existing'), ('existing', 'table'), ('table', '?'), ('?', '</s>')]


In [108]:
from nltk import FreqDist, bigrams
from collections import defaultdict

from collections import defaultdict
tokens_with_start_end = [['<s>'] + tokens + ['</s>'] for tokens in df_corpus['Tokens']]
all_tokens = [token for tokens in tokens_with_start_end for token in tokens]
fdist = FreqDist(all_tokens)
vocabb = {token: freq for token, freq in fdist.items() if freq >= 5}
restricted_vocab = set(vocabb.keys()).union({'<s>', '</s>', '<UNK>'}) 
print("restricted_vocab", len(restricted_vocab))

padded_sequence = [token for tokens in tokens_with_start_end for token in tokens]
print("padded_sequence", padded_sequence[:20])
padded_sequence_restricted = [token if token in restricted_vocab else '<UNK>' for token in padded_sequence]
padded_bigrams = list(bigrams(padded_sequence_restricted))
print("padded_sequence_restricted", padded_sequence_restricted[2])

bigram_freq = FreqDist(padded_bigrams)

start_token_freq = defaultdict(int)
for (first, second) in bigram_freq:
    if first == '<s>':
        start_token_freq[second] += bigram_freq[(first, second)]

total_start_occurrences = sum(start_token_freq.values())

start_token_probs = {token: count / total_start_occurrences for token, count in start_token_freq.items()}
top_4_tokens = dict(sorted(start_token_probs.items(), key=lambda x: x[1], reverse=True)[:4])

print(top_4_tokens)





restricted_vocab 19705
padded_sequence ['<s>', 'How', 'do', 'I', 'incorporate', 'a', 'for', '-', 'loop', 'into', 'this', 'function', 'and', 'still', 'have', 'the', 'same', 'outcomes', '?', 'I']
padded_sequence_restricted do
{'How': 0.1794568068364498, '<UNK>': 0.08562484135713681, 'how': 0.03426685844826127, 'What': 0.028851848718165665}


In [116]:
from nltk import FreqDist, bigrams
from collections import defaultdict
import numpy as np

# Step 1: Prepare the restricted vocabulary
tokens_with_start_end = [['<s>'] + tokens + ['</s>'] for tokens in df_corpus['Tokens']]
all_tokens = [token for tokens in tokens_with_start_end for token in tokens]
fdist = FreqDist(all_tokens)
vocabb = {token: freq for token, freq in fdist.items() if freq >= 5}
print("vocabb", vocabb)
restricted_vocab = set(vocabb.keys()).union({'<s>', '</s>', '<UNK>'}) 

# Step 2: Prepare padded sequences, replacing OOV words with <UNK>
padded_sequence_restricted = [token if token in restricted_vocab else '<UNK>' for token in all_tokens]
padded_bigrams = list(bigrams(padded_sequence_restricted))

# Step 3: Frequency distributions for bigrams and unigrams
bigram_freq = FreqDist(padded_bigrams)
unigram_freq = FreqDist(padded_sequence_restricted)

# Step 4: Define a function to compute Laplace-smoothed bigram probabilities
def laplace_bigram_prob(w1, w2, bigram_freq, unigram_freq, vocab_size):
    bigram_count = bigram_freq[(w1, w2)]
    unigram_count = unigram_freq[w1]
    return (bigram_count + 1) / (unigram_count + vocab_size)

# Step 5: Compute the top 4 tokens most likely to follow <s> (with Laplace smoothing)
vocab_size = len(restricted_vocab)
start_token_probs = {}
for token in restricted_vocab:
    prob = laplace_bigram_prob('<s>', token, bigram_freq, unigram_freq, vocab_size)
    start_token_probs[token] = prob

# Step 6: Get the top 4 tokens most likely to follow <s>
top_4_tokens = dict(sorted(start_token_probs.items(), key=lambda x: x[1], reverse=True)[:4])
print("Top 4 tokens likely to start a question:", top_4_tokens)

# Step 7: Define a function to compute perplexity
def compute_perplexity(test_corpus, bigram_freq, unigram_freq, restricted_vocab, vocab_size):
    total_log_prob = 0.0
    total_tokens = 0
    
    for tokens_question in test_corpus:
        tokens_question_restricted = [token if token in restricted_vocab else '<UNK>' for token in tokens_question]
        padded_question = ['<s>'] + tokens_question_restricted + ['</s>']
        question_bigrams = list(bigrams(padded_question))
        
        for (w1, w2) in question_bigrams[1:]:  # Exclude the first bigram with <s>
            prob = laplace_bigram_prob(w1, w2, bigram_freq, unigram_freq, vocab_size)
            total_log_prob += np.log(prob)
        
        total_tokens += len(tokens_question_restricted) + 1  # +1 for </s>
    
    perplexity = np.exp(-total_log_prob / total_tokens)
    return perplexity
df_corpus = pd.read_csv("./ressources/test.csv")

def remove_html_tags(text):
  import re
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)

df_corpus['Text'] = df_corpus['Text'].apply(lambda x: remove_html_tags(x))
df_corpus['Tokens'] = df_corpus['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))
# Step 8: Compute perplexity for the test set
test_corpus = df_corpus['Tokens'] # Tokenized test set

# Step 8: Compute perplexity for the test set
test_perplexity = compute_perplexity(test_corpus, bigram_freq, unigram_freq, restricted_vocab, vocab_size)
print(f"Test set perplexity: {test_perplexity:.3f}")


vocabb {'<s>': 11819, 'How': 4232, 'do': 4201, 'I': 34670, 'incorporate': 11, 'a': 24418, 'for': 10201, '-': 30508, 'loop': 882, 'into': 1581, 'this': 12697, 'function': 3602, 'and': 16385, 'still': 558, 'have': 7537, 'the': 39084, 'same': 1406, 'outcomes': 6, '?': 12020, 'written': 226, 'code': 5424, 'class': 4737, 'are': 2877, 'correct': 503, ',': 51942, 'but': 5982, 'did': 365, 'not': 6433, 'realize': 21, 'that': 8581, 'needed': 130, 'to': 35027, '.': 130553, 'The': 3118, 'exact': 102, 'words': 277, 'from': 6474, 'problem': 1443, '"': 27914, 'Hint': 5, 'use': 3247, 'variable': 1000, 'number': 1795, 'of': 12805, 'arguments': 194, '(*': 115, 'values': 1242, ')': 20959, 'access': 639, 'them': 916, 'with': 7574, '."': 213, 'don': 1229, "'": 26515, 't': 5088, 'really': 416, 'understand': 596, 'think': 386, 'all': 2448, 'need': 2219, 'is': 17248, 'add': 1661, 'somehow': 87, 'haven': 104, 'tried': 1496, 'anything': 333, 'yet': 152, 'because': 722, 'i': 12076, 'm': 4042, 'bad': 120, 'loops'

In [110]:
import numpy as np
from nltk import bigrams

def compute_bigram_probability(w1, w2, bigram_freq, unigram_freq, vocab_size):
    bigram_count = bigram_freq[(w1, w2)] 
    unigram_count = unigram_freq[w1]
    return (bigram_count + 1) / (unigram_count + vocab_size)

def compute_perplexity(test_corpus, bigram_freq, unigram_freq, restricted_vocab):
    vocab_size = len(restricted_vocab)
    total_log_prob = 0.0
    total_tokens = 0
    
    for tokens_question in test_corpus:
        tokens_question_restricted = [token if token in restricted_vocab else '<UNK>' for token in tokens_question]
        padded_question = ['<s>'] + tokens_question_restricted + ['</s>']
        question_bigrams = list(bigrams(padded_question))
        
        for (w1, w2) in question_bigrams[1:]:  
            prob = compute_bigram_probability(w1, w2, bigram_freq, unigram_freq, vocab_size)
            total_log_prob += np.log(prob)
        
        total_tokens += len(tokens_question_restricted) + 1 
    
    perplexity = np.exp(-total_log_prob / total_tokens)
    print("total_tokens", total_tokens)
    print("total_log_prob", total_log_prob)
    print("perplexity", perplexity)
    return perplexity

df_corpus = pd.read_csv("./ressources/train.csv")

def remove_html_tags(text):
  import re
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)

df_corpus['Text'] = df_corpus['Text'].apply(lambda x: remove_html_tags(x))
df_corpus['Tokens'] = df_corpus['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))

tokens = [token for sublist in df_corpus['Tokens'] for token in sublist]
fdist = FreqDist(tokens)
tokens_with_start_end = [['<s>'] + tokens + ['</s>'] for tokens in df_corpus['Tokens']]
all_tokens = [token for tokens in tokens_with_start_end for token in tokens]
fdist = FreqDist(all_tokens)
vocabb = {token: freq for token, freq in fdist.items() if freq >= 5}
restricted_vocab = set(vocabb.keys()).union({'<s>', '</s>', '<UNK>'}) 
print("restricted_vocab", len(restricted_vocab))

# Step 2: Prepare padded sequences, replacing OOV words with <UNK>
padded_sequence_restricted = [token if token in restricted_vocab else '<UNK>' for token in all_tokens]
padded_bigrams = list(bigrams(padded_sequence_restricted))

bigram_freq = FreqDist(padded_bigrams)
unigram_freq = FreqDist(padded_sequence_restricted)
df_corpus = pd.read_csv("./ressources/test.csv")

def remove_html_tags(text):
  import re
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)

df_corpus['Text'] = df_corpus['Text'].apply(lambda x: remove_html_tags(x))
df_corpus['Tokens'] = df_corpus['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))
test_corpus = df_corpus['Tokens']
test_perplexity = compute_perplexity(test_corpus, bigram_freq, unigram_freq, restricted_vocab)
print(f"Test set perplexity: {test_perplexity:.3f}")



restricted_vocab 19705
total_tokens 257918
total_log_prob -1650546.525724627
perplexity 601.5448546371044
Test set perplexity: 601.545


In [111]:
import pandas as pd
import numpy as np
from nltk import FreqDist
from nltk.tokenize import WordPunctTokenizer
from collections import Counter

df_corpus = pd.read_csv("./ressources/train.csv")

def preprocess_corpus(df):
    def remove_html_tags(text):
        import re
        clean = re.compile('<.*?>')
        return re.sub(clean, '', text)

    df['Text'] = df['Text'].apply(remove_html_tags)
    df['Tokens'] = df['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))
    return df

df_corpus = preprocess_corpus(df_corpus)

tokens = [token for sublist in df_corpus['Tokens'] for token in sublist]
fdist = FreqDist(tokens)

vocab = {token: freq for token, freq in fdist.items() if freq >= 5}
restricted_vocab = set(vocab.keys())

label_counts = Counter(df_corpus['Y'])

total_count = sum(label_counts.values())

prior_probs = {label: count / total_count for label, count in label_counts.items()}

def calculate_conditional_probabilities(df, vocab, prior_probs):
    class_word_counts = {label: Counter() for label in prior_probs.keys()}
    class_total_counts = {label: 0 for label in prior_probs.keys()}

    for index, row in df.iterrows():
        label = row['Y']
        tokens = row['Tokens']
        class_word_counts[label].update(token for token in tokens if token in vocab)
        class_total_counts[label] += len(tokens)

    conditional_probs = {}
    vocab_size = len(vocab)
    for label in class_word_counts:
        total_words = class_total_counts[label]
        for word in vocab:
            count = class_word_counts[label][word]
            smoothed_prob = (count + 1) / (total_words + vocab_size)
            conditional_probs[(word, label)] = smoothed_prob

    return conditional_probs

conditional_probs = calculate_conditional_probabilities(df_corpus, vocab, prior_probs)

def classify_question(tokens, prior_probs, conditional_probs):
    log_probs = {label: np.log(prior_probs[label]) for label in prior_probs}

    for token in tokens:
        if token in vocab:
            for label in prior_probs:
                log_probs[label] += np.log(conditional_probs.get((token, label), 1 / (total_count + len(vocab))))

    return max(log_probs, key=log_probs.get)

def evaluate_classifier(test_df, prior_probs, conditional_probs):
    correct = 0
    total = len(test_df)

    for index, row in test_df.iterrows():
        tokens = row['Tokens']
        actual_label = row['Y']
        predicted_label = classify_question(tokens, prior_probs, conditional_probs)

        if predicted_label == actual_label:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy

test_corpus = pd.read_csv("./ressources/test.csv") 
test_corpus = preprocess_corpus(test_corpus)

accuracy = evaluate_classifier(test_corpus, prior_probs, conditional_probs)
print(f"Accuracy of the trained classifier: {accuracy:.3f}%")


Accuracy of the trained classifier: 79.547%


In [112]:
import pandas as pd
import numpy as np
from nltk import FreqDist
from nltk.tokenize import WordPunctTokenizer
from collections import Counter

# Load the corpus
df_corpus = pd.read_csv("./ressources/train.csv")

# Function to remove HTML tags and tokenize
def preprocess_corpus(df):
    def remove_html_tags(text):
        import re
        clean = re.compile('<.*?>')
        return re.sub(clean, '', text)

    df['Text'] = df['Text'].apply(remove_html_tags)
    df['Tokens'] = df['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))
    return df

df_corpus = preprocess_corpus(df_corpus)

# Create a vocabulary and frequency distribution of tokens
tokens = [token for sublist in df_corpus['Tokens'] for token in sublist]
fdist = FreqDist(tokens)

# Create a restricted vocabulary based on frequency threshold
vocab = {token: freq for token, freq in fdist.items() if freq >= 5}
restricted_vocab = set(vocab.keys())

# Count occurrences of each label in the 'Y' column
label_counts = Counter(df_corpus['Y'])

# Calculate the total count of labels
total_count = sum(label_counts.values())

# Calculate prior probabilities P(c)
prior_probs = {label: count / total_count for label, count in label_counts.items()}

# Function to clip token counts to binary for each question
def clip_to_binary(tokens):
    return list(set(tokens))  # Converts list of tokens to unique set

# Apply binary clipping to each question in the corpus
df_corpus['Tokens_Binary'] = df_corpus['Tokens'].apply(clip_to_binary)

# Function to calculate binary conditional probabilities P(w_i | c) with Laplace smoothing
def calculate_conditional_probabilities_binary(df, vocab, prior_probs):
    # Initialize counts for each class
    class_word_counts = {label: Counter() for label in prior_probs.keys()}
    class_total_counts = {label: 0 for label in prior_probs.keys()}

    # Count unique words for each class
    for index, row in df.iterrows():
        label = row['Y']
        tokens = row['Tokens_Binary']  # Use the binary clipped tokens
        class_word_counts[label].update(token for token in tokens if token in vocab)
        class_total_counts[label] += len(tokens)

    # Calculate smoothed conditional probabilities P(w_i | c)
    conditional_probs = {}
    vocab_size = len(vocab)
    for label in class_word_counts:
        total_words = class_total_counts[label]
        for word in vocab:
            # Laplace smoothing
            count = class_word_counts[label][word]
            smoothed_prob = (count + 1) / (total_words + vocab_size)
            conditional_probs[(word, label)] = smoothed_prob

    return conditional_probs

conditional_probs_binary = calculate_conditional_probabilities_binary(df_corpus, vocab, prior_probs)

# Function to classify a single question using Binary Naive Bayes (logarithms for stability)
def classify_question_binary(tokens, prior_probs, conditional_probs):
    log_probs = {label: np.log(prior_probs[label]) for label in prior_probs}
    unique_tokens = set(tokens)  # Clip tokens to binary

    for token in unique_tokens:
        if token in vocab:
            for label in prior_probs:
                log_probs[label] += np.log(conditional_probs.get((token, label), 1 / (total_count + len(vocab))))

    return max(log_probs, key=log_probs.get)

# Classifying test corpus and calculating accuracy
def evaluate_classifier_binary(test_df, prior_probs, conditional_probs):
    correct = 0
    total = len(test_df)

    for index, row in test_df.iterrows():
        tokens = row['Tokens_Binary']
        actual_label = row['Y']
        predicted_label = classify_question_binary(tokens, prior_probs, conditional_probs)

        if predicted_label == actual_label:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy

# Load and preprocess the test corpus
test_corpus = pd.read_csv("./ressources/test.csv")  # Ensure your test data path is correct
test_corpus = preprocess_corpus(test_corpus)

# Apply binary clipping to the test data as well
test_corpus['Tokens_Binary'] = test_corpus['Tokens'].apply(clip_to_binary)

# Evaluate the Binary Naive Bayes classifier
accuracy_binary = evaluate_classifier_binary(test_corpus, prior_probs, conditional_probs_binary)

# Output the accuracy
print(f"Accuracy of the Binary Naive Bayes classifier: {accuracy_binary:.3f}%")


Accuracy of the Binary Naive Bayes classifier: 82.481%


In [113]:
import pandas as pd
import numpy as np
from nltk import FreqDist
from nltk.tokenize import WordPunctTokenizer
from collections import Counter
import re

# Load the corpus
df_corpus = pd.read_csv("./ressources/train.csv")

# Function to remove HTML tags
def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

# Preprocess the corpus: remove HTML tags and tokenize
def preprocess_corpus(df):
    df['Text'] = df['Text'].apply(remove_html_tags)
    df['Tokens'] = df['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))
    return df

df_corpus = preprocess_corpus(df_corpus)

# Define the list of negative tokens and punctuation marks
negation_words = ['not', 'no', 'never']
punctuation_marks = ['.', ',', ':', '?', '!']

# Function to replace tokens between negation and punctuation with t_NOT
def apply_negation(tokens):
    negated_tokens = []
    negation_active = False
    
    for token in tokens:
        if token in negation_words:
            negation_active = True
            negated_tokens.append(token)
        elif token in punctuation_marks:
            negation_active = False
            negated_tokens.append(token)
        elif negation_active:
            negated_tokens.append(f"{token}_NOT")
        else:
            negated_tokens.append(token)
    
    return negated_tokens

# Apply negation preprocessing to the Tokens column
df_corpus['Tokens_Negation'] = df_corpus['Tokens'].apply(apply_negation)

# Create a vocabulary and frequency distribution of tokens
tokens = [token for sublist in df_corpus['Tokens_Negation'] for token in sublist]
fdist = FreqDist(tokens)

# Create a restricted vocabulary based on frequency threshold
vocab = {token: freq for token, freq in fdist.items() if freq >= 5}
restricted_vocab = set(vocab.keys())

# Count occurrences of each label in the 'Y' column
label_counts = Counter(df_corpus['Y'])

# Calculate the total count of labels
total_count = sum(label_counts.values())

# Calculate prior probabilities P(c)
prior_probs = {label: count / total_count for label, count in label_counts.items()}

# Function to calculate conditional probabilities P(w_i | c) with Laplace smoothing
def calculate_conditional_probabilities(df, vocab, prior_probs):
    # Initialize counts for each class
    class_word_counts = {label: Counter() for label in prior_probs.keys()}
    class_total_counts = {label: 0 for label in prior_probs.keys()}

    # Count words for each class
    for index, row in df.iterrows():
        label = row['Y']
        tokens = row['Tokens_Negation']
        class_word_counts[label].update(token for token in tokens if token in vocab)
        class_total_counts[label] += len(tokens)

    # Calculate smoothed conditional probabilities P(w_i | c)
    conditional_probs = {}
    vocab_size = len(vocab)
    for label in class_word_counts:
        total_words = class_total_counts[label]
        for word in vocab:
            count = class_word_counts[label][word]
            smoothed_prob = (count + 1) / (total_words + vocab_size)
            conditional_probs[(word, label)] = smoothed_prob

    return conditional_probs

# Calculate the conditional probabilities for the modified tokens
conditional_probs_negation = calculate_conditional_probabilities(df_corpus, vocab, prior_probs)

# Function to classify a single question using Naive Bayes (logarithms for stability)
def classify_question(tokens, prior_probs, conditional_probs):
    log_probs = {label: np.log(prior_probs[label]) for label in prior_probs}

    for token in tokens:
        if token in vocab:
            for label in prior_probs:
                log_probs[label] += np.log(conditional_probs.get((token, label), 1 / (total_count + len(vocab))))

    return max(log_probs, key=log_probs.get)

# Classifying test corpus and calculating accuracy
def evaluate_classifier(test_df, prior_probs, conditional_probs):
    correct = 0
    total = len(test_df)

    for index, row in test_df.iterrows():
        tokens = row['Tokens_Negation']
        actual_label = row['Y']
        predicted_label = classify_question(tokens, prior_probs, conditional_probs)

        if predicted_label == actual_label:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy

# Load and preprocess the test corpus
test_corpus = pd.read_csv("./ressources/test.csv")  # Ensure your test data path is correct
test_corpus = preprocess_corpus(test_corpus)

# Apply negation preprocessing to the test data
test_corpus['Tokens_Negation'] = test_corpus['Tokens'].apply(apply_negation)

# Evaluate the Naive Bayes classifier with negation handling
accuracy_negation = evaluate_classifier(test_corpus, prior_probs, conditional_probs_negation)

# Output the accuracy
print(f"Accuracy of the Naive Bayes classifier with negation handling: {accuracy_negation:.3f}%")


Accuracy of the Naive Bayes classifier with negation handling: 79.883%


In [114]:
df_corpus_test = pd.read_csv("./ressources/test.csv")

def remove_html_tags(text):
  import re
  clean = re.compile('<.*?>')
  return re.sub(clean, '', text)

df_corpus_test['Text'] = df_corpus_test['Text'].apply(lambda x: remove_html_tags(x))
df_corpus_test['Tokens'] = df_corpus_test['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))

In [115]:
import math
from collections import defaultdict
from nltk.util import bigrams
from nltk import FreqDist

# Helper function to calculate bigram counts
def extract_bigrams(tokens_list, restricted_vocab):
    bigram_list = []
    for tokens in tokens_list:
        tokens_with_pad = ['<s>'] + [t if t in restricted_vocab else '<UNK>' for t in tokens] + ['</s>']
        bigram_list.extend(list(bigrams(tokens_with_pad)))
    return bigram_list

# Function to calculate perplexity for bigrams
def calculate_perplexity(bigram_freq, test_bigrams, vocab_size, epsilon, total_bigrams):
    log_prob_sum = 0.0
    N = len(test_bigrams)
    
    for bigram in test_bigrams:
        bigram_count = bigram_freq[bigram]
        bigram_probability = (bigram_count + epsilon) / (total_bigrams + epsilon * vocab_size)
        log_prob_sum += math.log(bigram_probability)
    
    perplexity = math.exp(-log_prob_sum / N)
    return perplexity

# Function to train bigram model with smoothing
def train_bigram_model(df_corpus, restricted_vocab, class_label):
    bigram_freq = defaultdict(int)
    total_bigrams = 0
    
    for tokens, label in zip(df_corpus['Tokens'], df_corpus['Y']):
        if label == class_label:
            tokens_with_pad = ['<s>'] + [t if t in restricted_vocab else '<UNK>' for t in tokens] + ['</s>']
            bigrams_class = list(bigrams(tokens_with_pad))
            total_bigrams += len(bigrams_class)
            for bigram in bigrams_class:
                bigram_freq[bigram] += 1
    
    return bigram_freq, total_bigrams

# Test with different epsilon values and choose the best
epsilons = [0.1, 0.01, 0.001, 0.0001]
best_epsilon = None
min_perplexity = float('inf')

# Training bigram models for all questions (class-unconditional)
all_bigrams = extract_bigrams(df_corpus['Tokens'], restricted_vocab)
all_bigram_freq = FreqDist(all_bigrams)
total_bigrams = sum(all_bigram_freq.values())

# Calculate perplexity for each epsilon
for epsilon in epsilons:
    perplexity = calculate_perplexity(all_bigram_freq, all_bigrams, len(restricted_vocab), epsilon, total_bigrams)
    print(f"Epsilon: {epsilon}, Perplexity: {perplexity:.3f}")
    
    if perplexity < min_perplexity:
        min_perplexity = perplexity
        best_epsilon = epsilon

print(f"Best epsilon: {best_epsilon}, Min perplexity: {min_perplexity:.3f}")

# Train class-conditional bigram models with the best epsilon
hq_bigram_freq, class_total_bigrams_HQ = train_bigram_model(df_corpus, restricted_vocab, 'HQ')
lq_bigram_freq, class_total_bigrams_LQ = train_bigram_model(df_corpus, restricted_vocab, 'LQ')

# Function to predict the class of a test question using bigrams
def predict_class(test_bigrams, hq_bigram_freq, lq_bigram_freq, class_total_bigrams_HQ, class_total_bigrams_LQ, best_epsilon, vocab_size):
    log_prob_HQ = 0.0
    log_prob_LQ = 0.0
    
    for bigram in test_bigrams:
        # Probability of the bigram in HQ
        bigram_count_HQ = hq_bigram_freq[bigram]
        prob_HQ = (bigram_count_HQ + best_epsilon) / (class_total_bigrams_HQ + best_epsilon * vocab_size)
        log_prob_HQ += math.log(prob_HQ)
        
        # Probability of the bigram in LQ
        bigram_count_LQ = lq_bigram_freq[bigram]
        prob_LQ = (bigram_count_LQ + best_epsilon) / (class_total_bigrams_LQ + best_epsilon * vocab_size)
        log_prob_LQ += math.log(prob_LQ)
    
    # Return the class with the higher log probability
    return 'HQ' if log_prob_HQ > log_prob_LQ else 'LQ'

# Evaluate the accuracy of the bigram classifier on the test set
correct_predictions = 0
total_predictions = 0

for tokens, true_label in zip(df_corpus_test['Tokens'], df_corpus_test['Y']):
    test_bigrams = list(bigrams(['<s>'] + [t if t in restricted_vocab else '<UNK>' for t in tokens] + ['</s>']))
    predicted_label = predict_class(test_bigrams, hq_bigram_freq, lq_bigram_freq, class_total_bigrams_HQ, class_total_bigrams_LQ, best_epsilon, len(restricted_vocab))
    
    if predicted_label == true_label:
        correct_predictions += 1
    
    total_predictions += 1

# Calculate accuracy
accuracy = correct_predictions / total_predictions * 100
print(f"Test set classification accuracy: {accuracy:.3f}%")


Epsilon: 0.1, Perplexity: 48571.634
Epsilon: 0.01, Perplexity: 49315.108
Epsilon: 0.001, Perplexity: 49392.803
Epsilon: 0.0001, Perplexity: 49400.608
Best epsilon: 0.1, Min perplexity: 48571.634
Test set classification accuracy: 82.146%


In [117]:
from nltk import FreqDist, bigrams
from collections import defaultdict
import numpy as np
import pandas as pd
from nltk.tokenize import WordPunctTokenizer

df_corpus = pd.read_csv("./ressources/train.csv")

def remove_html_tags(text):
    import re
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

df_corpus['Text'] = df_corpus['Text'].apply(lambda x: remove_html_tags(x))
df_corpus['Tokens'] = df_corpus['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))

tokens_with_start_end = [['<s>'] + tokens + ['</s>'] for tokens in df_corpus['Tokens']]
all_tokens = [token for tokens in tokens_with_start_end for token in tokens]
fdist = FreqDist(all_tokens)
vocabb = {token: freq for token, freq in fdist.items() if freq >= 5}
restricted_vocab = set(vocabb.keys()).union({'<s>', '</s>', '<UNK>'})

padded_sequence_restricted = [token if token in restricted_vocab else '<UNK>' for token in all_tokens]
padded_bigrams = list(bigrams(padded_sequence_restricted))

bigram_freq = FreqDist(padded_bigrams)
unigram_freq = FreqDist(padded_sequence_restricted)

def laplace_bigram_prob(w1, w2, bigram_freq, unigram_freq, vocab_size):
    bigram_count = bigram_freq[(w1, w2)]
    unigram_count = unigram_freq[w1]
    return (bigram_count + 1) / (unigram_count + vocab_size)

vocab_size = len(restricted_vocab)
start_token_probs = {}
for token in restricted_vocab:
    prob = laplace_bigram_prob('<s>', token, bigram_freq, unigram_freq, vocab_size)
    start_token_probs[token] = prob

top_4_tokens = dict(sorted(start_token_probs.items(), key=lambda x: x[1], reverse=True)[:4])
print("Top 4 tokens:", top_4_tokens)

def compute_perplexity(test_corpus, bigram_freq, unigram_freq, restricted_vocab, vocab_size):
    total_log_prob = 0.0
    total_tokens = 0
    
    for tokens_question in test_corpus:
        tokens_question_restricted = [token if token in restricted_vocab else '<UNK>' for token in tokens_question]
        padded_question = ['<s>'] + tokens_question_restricted + ['</s>']
        question_bigrams = list(bigrams(padded_question))
        
        for (w1, w2) in question_bigrams[1:]:
            prob = laplace_bigram_prob(w1, w2, bigram_freq, unigram_freq, vocab_size)
            total_log_prob += np.log(prob)
        total_tokens += len(tokens_question_restricted) 
    perplexity = np.exp(-total_log_prob / total_tokens)
    return perplexity

df_test = pd.read_csv("./ressources/test.csv")
df_test['Text'] = df_test['Text'].apply(lambda x: remove_html_tags(x))
df_test['Tokens'] = df_test['Text'].apply(lambda x: WordPunctTokenizer().tokenize(x))

test_corpus = df_test['Tokens']
test_perplexity = compute_perplexity(test_corpus, bigram_freq, unigram_freq, restricted_vocab, vocab_size)
print(f"perplexity: {test_perplexity:.3f}")


Top 4 tokens: {'How': 0.06731379266590534, '<UNK>': 0.032134246922979316, 'how': 0.01287907625935795, 'What': 0.010848877046060145}
perplexity: 619.703
